# 3.1 - What Is Generative AI?

**Module 3 - GenAI Foundations** | Netsetos GenAI Engineering

This notebook makes the core ideas of generative AI concrete by running them: it contrasts a discriminative classifier with a generative model, turns the temperature dial, fires one model at six different tasks, and provokes the four classic LLM failure modes on purpose. It closes with the three ways to customize a model (prompting, RAG, fine-tuning), a 2026 model-landscape table, and the Chat to RAG to Agents progression the rest of the course builds on.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Environment prep for the whole notebook. The install line is commented out for local runs; uncomment it in Colab or a fresh environment. Nothing here calls a model yet - it just names the packages the later cells rely on.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai scikit-learn -q google-genai

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line prep cell. It documents the `pip install` you may need and leaves a note that the lesson uses seeded random values for reproducibility - there is no executable logic to run.

**How the code works - step by step**
- **Commented `pip install`** - pulls `anthropic`, `openai`, `scikit-learn`, and `google-genai`; the Gemini calls below need `google-genai`, and the first demo needs `scikit-learn`.
- **Reproducibility comment** - flags that seeded values keep runs comparable.

**In one line:** install the libraries, then move on.

**What you'll see:** (no output - environment prep)

## 1 - Discriminative AI: the judge

Before you can appreciate what generative AI adds, you need to feel what came before it. This cell trains a tiny sentiment classifier on six movie reviews and asks it to label a seventh - a model that can only sort inputs into buckets, never produce new text. That one-word ceiling is exactly the contrast the next cell breaks.

In [ ]:
# =============================================
# DISCRIMINATIVE AI - The Judge
# Input: movie review → Output: "positive" or "negative"
# It classifies. It NEVER creates new text.
# =============================================

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Training data
reviews = [
    "This movie was absolutely fantastic! Loved every minute.",
    "Terrible film. Waste of time and money.",
    "A masterpiece! The acting was incredible.",
    "Boring and predictable. Would not recommend.",
    "One of the best films I've ever seen!",
    "Awful screenplay. The plot made no sense.",
]
labels = ["positive", "negative", "positive", "negative", "positive", "negative"]

# Train
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(reviews)
model = MultinomialNB()
model.fit(X, labels)

# Classify a NEW review
new_review = "The movie was engaging and well-directed."
prediction = model.predict(vectorizer.transform([new_review]))[0]

print(f"Review:  '{new_review}'")
print(f"AI says: {prediction}")
print(f"Can it WRITE a review? ❌ No. It only judges.")
# Output:
# Review:  'The movie was engaging and well-directed.'
# AI says: positive
# Can it WRITE a review? No - it only judges.

**Explanation**

A complete, self-contained classical-ML pipeline that runs locally with no API key. It turns text into word counts, fits a Naive Bayes classifier, and predicts a single label for an unseen review. The point is the ceiling: its entire output is one word from a fixed set - it recognizes, it does not create.

**How the code works - step by step**
- **`reviews` / `labels`** - six hand-labeled training examples, three positive and three negative.
- **`CountVectorizer().fit_transform`** - converts each review into a bag-of-words count vector `X`.
- **`MultinomialNB().fit(X, labels)`** - trains a Naive Bayes classifier on those vectors.
- **`model.predict(vectorizer.transform([new_review]))`** - vectorizes a brand-new review and returns its predicted label.
- **`print(...)`** - shows the review, the verdict, and the punchline that it cannot write a review.

**In one line:** count words -> train Naive Bayes -> emit one label, never new text.

**What you'll see:** The new review prints, then `AI says: positive`, then the line noting it can only judge, not write.

## 2 - Generative AI: the creator

Same problem space, opposite capability. This cell sends three prompts to Gemini and gets back three things that never existed before: a movie review, a working function, and an analogy. Where the classifier emitted one word, the generator writes whole passages.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# GENERATIVE AI - The Creator
# Input: a prompt → Output: NEW text that never existed
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# Task 1: CREATE a movie review (not classify - CREATE)
resp = client.models.generate_content(model="gemini-2.5-flash", contents=
    "Write a 3-sentence review for a fictional Bollywood "
    "sci-fi film called 'Mars Wala Love'."
)
print("Generated review:")
print(f"  {resp.text}\n")

# Task 2: CREATE code
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Write a Python function to check if a number is prime.")
print("Generated code:")
print(f"  {resp.text}\n")

# Task 3: CREATE an analogy
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Explain WiFi using the analogy of a chai stall in a busy market.")
print("Generated analogy:")
print(f"  {resp.text}")
# Output:  (generated text varies by run - example below)
# Generated review:
#   Mars Wala Love is a dazzling ride that fuses Bollywood romance with hard sci-fi.
# Generated code:
#   def is_prime(n): return n > 1 and all(n % i for i in range(2, int(n**0.5)+1))
# Generated analogy:
#   WiFi is like a chai stall taking orders shouted across a busy market...

**Explanation**

The first real model call. It creates a `genai.Client()` and makes three independent `generate_content` calls with `gemini-2.5-flash`, each producing novel text. This is the concrete demonstration that generation is a superset of classification - it can create, and creating is strictly harder than judging.

**How the code works - step by step**
- **`client = genai.Client()`** - constructs the client, reading the API key from the environment.
- **Task 1 (review)** - prompts for a 3-sentence review of a fictional film and prints `resp.text`.
- **Task 2 (code)** - prompts for a prime-checking Python function and prints it.
- **Task 3 (analogy)** - prompts for a WiFi-as-chai-stall analogy and prints it.

**In one line:** one client, three prompts, three pieces of brand-new content.

**What you'll see:** Three labeled blocks - Generated review, Generated code, Generated analogy - each with model-written text that varies run to run (the sample output shows a one-line example of each).

## 3 - Temperature: the creativity dial

Generation is next-word prediction, and temperature decides how boldly the model picks that next word. This cell sends one identical prompt at four temperatures and prints the four continuations side by side, so you can watch low temperature stay safe and high temperature go wild.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# TEMPERATURE EXPERIMENT
# Same prompt, different creativity levels.
# =============================================

from google import genai
from google.genai import types

client = genai.Client()

prompt = "Continue in 1 sentence: 'The robot opened its eyes for the first time and saw'"

for temp in [0.0, 0.5, 1.0, 1.5]:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=40))
    labels = {0.0: "📏 Predictable", 0.5: "⚖️ Balanced",
              1.0: "🎨 Creative", 1.5: "🌪️ Wild"}
    print(f"  temp={temp} {labels[temp]}:")
    print(f"    {resp.text.strip()[:100]}\n")
# Output:  (varies by run)
#   temp=0.0  Predictable:  a vast, silent city of steel and glass.
#   temp=0.5  Balanced:     a world reborn in light and shadow.
#   temp=1.0  Creative:     oceans of liquid starlight under three moons.
#   temp=1.5  Wild:         clockwork birds reciting half-remembered names.

**Explanation**

A controlled experiment: hold the prompt fixed, sweep one knob. It loops over four temperature values, passing each through `GenerateContentConfig`, and prints the labeled result. Read it as an intuition-builder for what the temperature parameter actually does to output.

**How the code works - step by step**
- **`prompt`** - a single fixed sentence-completion prompt, reused every iteration.
- **`for temp in [0.0, 0.5, 1.0, 1.5]`** - sweeps from deterministic to highly random.
- **`config=types.GenerateContentConfig(temperature=temp, max_output_tokens=40)`** - sets the creativity level and caps length per call.
- **`labels` dict + `print`** - tags each run (Predictable / Balanced / Creative / Wild) and prints the first 100 chars.

**In one line:** same prompt, four temperatures -> watch randomness climb.

**What you'll see:** Four labeled lines, temp=0.0 through temp=1.5, each with a progressively more surprising sentence continuation (text varies by run).

## 4 - One model, six tasks

Generative AI is not a chatbot with one trick - it is a general creative engine. This cell drives the same Gemini model through six unrelated jobs (story, code, structured data, translation, advice, role-play) to show the breadth comes from the prompt, not from six specialized models.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# ONE MODEL, SIX TASKS
# The same Gemini model generates all of these.
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

tasks = [
    ("📖 Story",       "Tell me a 3-line horror story set in a Hyderabad IT office at 3 AM."),
    ("💻 Code",        "Write a Python function that checks if a string is a palindrome."),
    ("📊 Data",        "Generate JSON: 3 fictional Indian startups with name, city, funding_crores."),
    ("🌐 Translation", "Translate 'Knowledge is power' into Hindi, Telugu, and Tamil."),
    ("🎓 Advice",      "Should a B.Tech CSE graduate learn GenAI or Data Science first? 2 sentences each."),
    ("👵 Role-play",   "You're an Indian grandmother. Explain what an API is using a chai-making analogy."),
]

for label, prompt in tasks:
    print(f"\n{'═'*50}\n{label}\n{'═'*50}")
    print(client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip())
# Output:  (6 tasks, one model - text varies by run)
#   Story:       The server room hummed at 3 AM when the cursor moved on its own...
#   Code:        def is_palindrome(s): return s == s[::-1]
#   Data:        [{'name': 'ChaiByte', 'city': 'Pune', 'funding_crores': 12}, ...]
#   Translation: knowledge-is-power rendered in Hindi, Telugu, and Tamil
#   Advice:      Learn GenAI first if you enjoy product-building; DS if you like stats...
#   Role-play:   Beta, an API is like asking the chaiwala for chai - you don't boil milk yourself...

**Explanation**

A breadth demo built as a simple loop over a list of (label, prompt) pairs. Every task hits the identical `gemini-2.5-flash` model - only the instruction changes. The takeaway is that task variety is a prompting choice, not a model-selection choice.

**How the code works - step by step**
- **`tasks`** - a list of six (label, prompt) tuples spanning story, code, JSON data, translation, advice, and role-play.
- **`for label, prompt in tasks`** - iterates over all six.
- **`generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()`** - runs each prompt through the same model and prints the result under a divider.

**In one line:** one model, six prompts -> six completely different outputs.

**What you'll see:** Six divider-separated sections (Story, Code, Data, Translation, Advice, Role-play), each with model output for that task (text varies by run).

## 5 - The scale effect

One of the three breakthroughs behind modern GenAI is sheer scale: bigger models reason better. This cell sends the same explain-to-a-10-year-old prompt to a smaller model and a larger one so you can compare the quality jump directly.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# THE SCALE EFFECT
# Same prompt → different model sizes.
# Larger model = better output quality.
# =============================================

from google import genai

client = genai.Client()

prompt = "Explain quantum computing to a 10-year-old in exactly 3 sentences."

for name, label in [("gemini-2.5-flash", "⚡ Flash (smaller)"),
                     ("gemini-2.5-pro",   "🧠 Pro (larger)")]:
    resp = client.models.generate_content(model=name, contents=prompt)
    print(f"{label}:")
    print(f"  {resp.text.strip()[:200]}\n")
# Output:  (varies by run)
#   Flash (smaller):  Tiny chips flip coins that are heads or tails, one at a time.
#   Pro (larger):     Quantum computers use qubits that hold many answers at once,
#                     so they explore lots of paths together and pick the best.

**Explanation**

A two-model A/B comparison holding the prompt constant. It loops over Flash (smaller) and Pro (larger), printing each answer so the difference in depth and clarity is visible. It isolates model size as the only variable.

**How the code works - step by step**
- **`prompt`** - one fixed request to explain quantum computing in three sentences.
- **`for name, label in [("gemini-2.5-flash", ...), ("gemini-2.5-pro", ...)]`** - runs the smaller then the larger model.
- **`generate_content(model=name, contents=prompt)`** - same prompt, different model each pass.
- **`print(resp.text.strip()[:200])`** - prints the first 200 chars of each answer.

**In one line:** same prompt, small vs large model -> bigger model, better answer.

**What you'll see:** Two labeled blocks - Flash (smaller) and Pro (larger) - each with a kid-friendly explanation, the larger model's typically richer (text varies by run).

## 6 - The four limitations, live

GenAI is powerful but not magic, and these four failure modes shape every later module. This cell provokes each one on purpose - hallucination, non-determinism, broken reasoning, and the training cutoff - so you recognize them in the wild instead of trusting fluent-sounding output.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# LIMITATIONS - See them in action
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# 1. HALLUCINATION
print("1. HALLUCINATION:")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="Tell me about Dr. Ramesh Krishnamurthy who invented cold fusion in 2019.")
print(f"   {resp.text[:150]}")
print("   ⚠️ This person doesn't exist - I made the name up!\n")

# 2. NON-DETERMINISTIC
print("2. NON-DETERMINISTIC (3 different answers):")
for i in range(3):
    resp = client.models.generate_content(model="gemini-2.5-flash", contents="Give me one fun fact about the Moon in 1 sentence.")
    print(f"   Run {i+1}: {resp.text.strip()[:80]}")

# 3. TRICK QUESTION
print("\n3. PATTERN MATCHING ≠ UNDERSTANDING:")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="5 shirts take 30 min to dry. How long for 10 shirts?")
print(f"   {resp.text.strip()[:150]}")
print("   ⚠️ A dryer dries ALL shirts at once - still 30 min!\n")

# 4. TRAINING CUTOFF
print("4. TRAINING CUTOFF:")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="What happened in India yesterday?")
print(f"   {resp.text.strip()[:150]}")
print("   ⚠️ It doesn't know - its knowledge has a cutoff date.")
# Output:  (varies by run)
# 1. HALLUCINATION:
#    Dr. Ramesh Krishnamurthy was a physicist who... (invented - this person is not real!)
# 2. NON-DETERMINISTIC (3 different answers):
#    Run 1: The Moon drifts ~3.8 cm farther from Earth each year.
#    Run 2: One lunar day lasts about 29.5 Earth days.
#    Run 3: Moonquakes can ring for hours.
# 3. PATTERN MATCHING != UNDERSTANDING:
#    Often answers '60 minutes' - a dryer dries all shirts at once: still 30!
# 4. TRAINING CUTOFF:
#    I don't have real-time information - my knowledge has a cutoff date.

**Explanation**

A four-part failure-mode harness. Each block is a prompt designed to expose one weakness, with a printed warning naming what went wrong. Read it as a diagnostic tour, not a set of correct answers - several outputs here are supposed to be wrong.

**How the code works - step by step**
- **1. Hallucination** - asks about a person the prompt invented; the model tends to describe them as real anyway.
- **2. Non-determinism** - runs the same Moon-fact prompt three times in a loop to show three different answers.
- **3. Pattern matching != understanding** - the drying-shirts trick question, where the model often multiplies instead of reasoning that a dryer dries all shirts at once.
- **4. Training cutoff** - asks about yesterday's news, which the model cannot know.
- **`print(... warning)`** - after each, a line explaining why the output is unreliable.

**In one line:** four prompts, four built-in weaknesses exposed on purpose.

**What you'll see:** Four numbered sections, each printing the model's answer plus a warning line - a fabricated bio, three differing Moon facts, a likely-wrong shirt-drying answer, and a no-real-time-info reply (text varies by run).

## 7 - Three ways to customize an LLM

You do not build foundation models - you customize them, and there are exactly three levers. This cell demonstrates prompting and RAG live, then describes fine-tuning, and prints the decision framework for choosing between them.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# THREE WAYS TO CUSTOMIZE AN LLM
# 1. Prompting: cheapest, fastest, least control
# 2. RAG: add your own knowledge at query time
# 3. Fine-tuning: retrain on your data
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# ── 1. PROMPTING - Change behavior with instructions ──
# Free, instant. Limited by what the model already knows.
print("1. PROMPTING (zero cost):")
resp = client.models.generate_content(model="gemini-2.5-flash", contents=
    "You are a senior Python developer. Review this code: "
    "def add(a, b): return a + b\n"
    "Give 3 improvement suggestions."
)
print(f"   {resp.text.strip()[:150]}\n")

# ── 2. RAG - Inject your own documents ──
# Your data goes into the CONTEXT, not the model weights.
print("2. RAG (your data + model = accurate answers):")
company_docs = """
NETSETOS REFUND POLICY (2024):
- Full refund within 7 days of purchase.
- 50% refund within 30 days.
- No refund after 30 days.
- Contact: support@netsetos.com
"""
resp = client.models.generate_content(model="gemini-2.5-flash", contents=
    f"Based ONLY on this document:\n{company_docs}\n\n"
    f"Question: Can I get a refund after 15 days?"
)
print(f"   {resp.text.strip()[:150]}\n")

# ── 3. FINE-TUNING - Retrain the model ──
# Changes the model's weights. Expensive but powerful.
print("3. FINE-TUNING (changes model behavior permanently):")
print("   Example: Train Gemini on 10,000 customer support tickets")
print("   Result: Model learns your company's tone, policies, jargon")
print("   Cost: ~$50-500 + 1-4 hours of training time")
print("   When: Only after prompting + RAG aren't enough (Module 9)\n")

print("Decision framework:")
print("  Start with prompting (free, 5 min)")
print("  Add RAG if model needs YOUR data (Module 8)")
print("  Fine-tune only if behavior must change fundamentally (Module 9)")
# Output:  (varies by run)
# 1. PROMPTING (zero cost):
#    1) Add type hints  2) Add a docstring  3) Guard against non-numeric input...
# 2. RAG (your data + model = accurate answers):
#    Per the policy, a refund after 15 days is 50% (it falls within 30 days).
# 3. FINE-TUNING (changes model behavior permanently):
#    Example: train on 10,000 support tickets -> learns your tone, policies, jargon.
# Decision framework:
#    Start with prompting -> add RAG if it needs your data -> fine-tune last.

**Explanation**

A spectrum walkthrough, part live and part narrated. Prompting and RAG are real model calls; fine-tuning is described with print statements (it is a Module 9 topic). The core idea: prompting changes behavior with words, RAG feeds your data into the context, fine-tuning rewrites the weights - increasing in cost and control.

**How the code works - step by step**
- **1. Prompting** - a system-style instruction ("You are a senior Python developer...") plus a code-review request; a real call, zero extra cost beyond the token bill.
- **2. RAG** - stuffs a `company_docs` refund policy string into the prompt and asks a question answerable only from it, so the model answers from YOUR data.
- **3. Fine-tuning** - printed description only: what it does, rough cost, and when to reach for it.
- **Decision framework** - final prints: start with prompting, add RAG if the model needs your data, fine-tune last.

**In one line:** prompt (free) -> RAG (your data) -> fine-tune (retrain), cheapest first.

**What you'll see:** Three numbered sections - a prompted code review, a RAG answer grounded in the refund policy, and a printed fine-tuning summary - followed by the decision-framework lines (model text varies by run).

## 8 - The 2026 model landscape

Context windows and per-token price are the two numbers that drive most model-choice decisions. This cell is a pure reference table - no API call - laying out six leading models by provider, context size, input cost, and whether they are open-source, then printing when to pick each.

In [ ]:
# =============================================
# THE 2026 MODEL LANDSCAPE
# Context windows, costs, and when to use each
# =============================================

models = [
    # (Name, Provider, Context, Input$/M, Open-source?)
    # Illustrative sketch of the 2026 landscape - model names, prices, and
    # context windows are approximate and move fast; check current provider docs.
    ("Gemini 2.5 Flash",    "Google",    1_000_000,  0.30,  False),
    ("GPT-5",                "OpenAI",      400_000,  2.50,  False),
    ("Claude Sonnet 4.6",   "Anthropic",   1_000_000,  3.00,  False),
    ("Llama 4 Maverick",      "Meta",        1_000_000,  0.00,  True),
    ("Mistral Large",        "Mistral",     256_000,  2.00,  False),
    ("DeepSeek V3",          "DeepSeek",    128_000,  0.27,  True),
]

print(f"{'Model':<22s} {'Provider':<12s} {'Context':>10s} {'$/M Input':>10s} {'Open?':>6s}")
print("-" * 65)
for name, provider, ctx, cost, oss in models:
    ctx_str = f"{ctx//1000}K"
    cost_str = "FREE" if cost == 0 else f"${cost:.2f}"
    oss_str = "Yes" if oss else "No"
    print(f"  {name:<20s} {provider:<12s} {ctx_str:>10s} {cost_str:>10s} {oss_str:>6s}")

print(f"""
Key decisions:
  Gemini Flash  → Default choice. Cheapest, 1M context, multimodal.
  GPT-5         → When you need the OpenAI ecosystem (function calling, assistants).
  Claude Sonnet → Best for writing, long docs, code generation.
  LLaMA/DeepSeek → Self-host for privacy, no API costs, full control.

Open-source = you can run it on YOUR servers.
Closed-source = API only, vendor controls the model.
Module 13 covers self-hosting with Ollama and vLLM.
""")
# Output:
# Model                  Provider        Context  $/M Input  Open?
# -----------------------------------------------------------------
#   Gemini 2.5 Flash     Google            1000K      $0.30     No
#   GPT-5                OpenAI             400K      $2.50     No
#   Claude Sonnet 4.6    Anthropic         1000K      $3.00     No
#   Llama 4 Maverick     Meta              1000K       FREE    Yes
#   Mistral Large        Mistral            256K      $2.00     No
#   DeepSeek V3          DeepSeek           128K      $0.27    Yes

**Explanation**

A local formatting cell, not a model call. It holds an illustrative list of models and prints a neatly aligned comparison table plus a decision guide. Treat the exact numbers as approximate and fast-moving - the value is the shape of the trade-offs, not the precise prices.

**How the code works - step by step**
- **`models`** - a list of (name, provider, context, input $/M, open-source?) tuples for six 2026 models.
- **Header + `print("-" * 65)`** - prints the aligned column headers and a rule.
- **`for ... in models`** - formats each row: context as `K`, cost as `FREE` or a dollar figure, open-source as Yes/No.
- **Final `print(f"""...""")`** - a key-decisions block on when to use each model and what open vs closed means.

**In one line:** a hard-coded reference table of the 2026 model options and their trade-offs.

**What you'll see:** A formatted six-row table (Gemini Flash, GPT-5, Claude Sonnet 4.6, Llama 4 Maverick, Mistral Large, DeepSeek V3) with context, input cost, and open-source flag, followed by a key-decisions guide. Deterministic - no model call, so output matches the `# Output:` block.

## 9 - Beyond chat: RAG and agents

The course roadmap in one cell. A plain LLM answers only from memory; RAG adds a knowledge bookshelf; agents add tools that let it act. This cell runs the plain-LLM case live and then narrates the RAG and agent pipelines to show the Chat to RAG to Agents progression.

> **Needs a Gemini API key** (set `GEMINI_API_KEY` or `GOOGLE_API_KEY`).

In [ ]:
# =============================================
# RAG: Give the LLM YOUR knowledge
# AGENTS: Give the LLM TOOLS
# =============================================

from google import genai

# Client() reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment
client = genai.Client()

# ── 1. PLAIN LLM - Limited to training data ──
print("1. PLAIN LLM (no knowledge of your company):")
resp = client.models.generate_content(model="gemini-2.5-flash", contents="What is Netsetos's refund policy?")
print(f"   {resp.text.strip()[:120]}")
print("   ⚠️ It doesn't know - will hallucinate or say 'I don't know'\n")

# ── 2. RAG - Fetch docs, stuff into context ──
print("2. RAG (Retrieval-Augmented Generation):")
print("   Step 1: User asks 'What's the refund policy?'")
print("   Step 2: System searches your document database")
print("   Step 3: Relevant policy document is RETRIEVED")
print("   Step 4: Document is stuffed into the CONTEXT")
print("   Step 5: LLM generates answer FROM the document")
print("   → Accurate, grounded, no hallucination!")
print("   → Module 8 builds this complete pipeline\n")

# ── 3. AGENTS - LLM + Tools ──
print("3. AGENTS (LLM + Tools = Actions):")
print("   User: 'Book me a flight to Mumbai tomorrow under ₹5000'")
print("   Agent thinks: I need to search flights → compare prices → book")
print("   Step 1: Calls flight search API (tool)")
print("   Step 2: Filters results under ₹5000 (reasoning)")
print("   Step 3: Calls booking API (tool)")
print("   Step 4: Confirms with user (human-in-the-loop)")
print("   → The LLM didn't just answer - it ACTED!")
print("   → Module 11 builds multi-step agents\n")

print("The evolution of GenAI applications:")
print("  Chat (Module 3)  →  RAG (Module 8)  →  Agents (Module 11)")
print("  'Answer from memory'  'Answer from docs'  'Take actions'")
# Output:
# 1. PLAIN LLM (no knowledge of your company):
#    I don't have specific details on Netsetos's refund policy... (may hallucinate)
# 2. RAG (Retrieval-Augmented Generation):
#    Steps 1-5 retrieve the policy doc -> Accurate, grounded, no hallucination!
# 3. AGENTS (LLM + Tools = Actions):
#    Search flights -> filter under 5000 -> book -> confirm. It didn't just answer; it ACTED!

**Explanation**

A roadmap cell, part live and part narrated. Only the first block calls the model - to show it does not know a private company's policy; the RAG and agent sections are printed step-by-step descriptions of pipelines built later in the course. The idea is the three-rung ladder: answer from memory, answer from your docs, take actions.

**How the code works - step by step**
- **1. Plain LLM** - a real call asking for Netsetos's refund policy, which the base model cannot know and may hallucinate.
- **2. RAG** - printed five-step pipeline: ask -> search docs -> retrieve -> stuff into context -> generate a grounded answer (built in Module 8).
- **3. Agents** - printed multi-step flight-booking scenario: search API -> filter -> book -> confirm, where the model acts, not just answers (built in Module 11).
- **Evolution line** - final prints mapping Chat (M3) -> RAG (M8) -> Agents (M11).

**In one line:** memory -> retrieved docs -> tools and actions, the course arc in three rungs.

**What you'll see:** Three sections - a live (and unreliable) plain-LLM answer about the refund policy, then printed RAG and agent pipelines, closing with the Chat -> RAG -> Agents evolution lines (the first answer varies by run; the rest is fixed text).

Across ten cells you saw the whole shape of the field: discriminative AI judges, generative AI creates, and everything an LLM does is next-token prediction with a temperature knob on the randomness. You watched one model handle six tasks, saw quality scale with model size, and deliberately triggered hallucination, non-determinism, bad reasoning, and the training cutoff so you know the failure modes before you build on them. The customization spectrum (prompting to RAG to fine-tuning), the model-landscape table, and the Chat to RAG to Agents ladder are the map for the rest of the course - Module 5 deepens prompting, Module 8 builds RAG, and Modules 10-11 build agents.